In [1]:
import yaml
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Leer credenciales del arhivo YAML
with open("credenciales.yaml", "r") as file:
     config = yaml.safe_load(file)

db = config["mysql"]

# Crear conexión
# Su función es crear un Engine, que es el objeto que SQLAlchemy utiliza para comunicarse con la base de datos.
engine = create_engine(                                     
    f"mysql+pymysql://{db['user']}:{db['password']}@"
    f"{db['host']}:{db['port']}/{db['database']}"
)

##### 4.- ANALISIS DE COHORTE

##### 4-1 Introduccion a las Cohortes

<span style="font-size: 12px;">

Agrupamiento generacional: Cual es caracteristica para agrupar gente en la cohorte, por ejemplo: fecha de nacimiento, primer compra de un producto, nacionalidad, género, sexo. No debe cambiar a lo largo del estudio.

Temporalidad: Tiempo en que se estará siguiendo al grupo.

Métrica agregada: Cómo mediremos a las personas de la cohorte, pueden ser ventas, ticket promedio, calificaciones, asistencia a clase, tasa de clicks, etc.


Temas:

Análisis de retención: Si cada miembro de una generación está presenta en una fecha en particular de nuestro estudio.

Análisis de superviviencia: Cuanta gente de una generación sobrevive después de cierto tiempo.

Análisis de retorno: Si una acción ha sucedido más de n veces en un periodo de tiempo dado. Por ejemplo, cuantas veces nos compra un cliente en un periodo de tiempo.

Análisis Acumulativos: A lo largo de la vida del cliente, cuento dinero ha gastado con nosotros.

</span>


##### 4-2 Introducción a Análisis de Retencion

<span style="font-size: 12px;">

Qué tanta gente se queda con nosotros. Ver el tamaño de la cohorte a través del tiempo, o bien, que tanto dinero gasta la gente a traves del tiempo. 

Buscamos expresar como un %. Al inicio se tiene un 100%, luego puede aumentar o disminuir.

Se puede considerar el número de personas o gasto de dinero a través del tiempo.

</span>

##### 4-3 SQL para curva basica de Retencion

In [21]:
query="""
SELECT
     * 
FROM
     cohort.supertienda
LIMIT
     10;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,venta_id,Categoria,Ciudad,Pais/Región,Nombre_del_Cliente,Proveedor,Fecha_de_Orden,ID_Orden,Codigo_Postal,Nombre_de_Producto,Region,Segmento,Fecha_de_envio,Estado/Provincia,Descuento,Ganancia,Cantidad,Ventas
0,1,Suministros de Oficina,Houston,United States,Darren Powers,Message Book,2019-01-03,US-2019-103800,77095,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",Central,Consumidor,2019-01-07,Texas,0.2,5.5512,2,16.448
1,2,Suministros de Oficina,Naperville,United States,Phillina Ober,GBC,2019-01-04,US-2019-112326,60540,GBC Standard Plastic Binding Systems Combs,Central,Home Office,2019-01-08,Illinois,0.8,-5.4870,2,3.540
2,3,Suministros de Oficina,Naperville,United States,Phillina Ober,Avery,2019-01-04,US-2019-112326,60540,Avery 508,Central,Home Office,2019-01-08,Illinois,0.2,4.2717,3,11.784
3,4,Suministros de Oficina,Naperville,United States,Phillina Ober,SAFCO,2019-01-04,US-2019-112326,60540,SAFCO Boltless Steel Shelving,Central,Home Office,2019-01-08,Illinois,0.2,-64.7748,3,272.736
4,5,Suministros de Oficina,Philadelphia,United States,Mick Brown,Avery,2019-01-05,US-2019-141817,19143,Avery Hi-Liter EverBold Pen Style Fluorescent ...,Este,Consumidor,2019-01-12,Pennsylvania,0.2,4.8840,3,19.536
5,6,Muebles,Henderson,United States,Maria Etezadi,Global,2019-01-06,US-2019-167199,42420,Global Deluxe High-Back Manager's Chair,Sur,Home Office,2019-01-10,Kentucky,0.0,746.4078,9,2573.820
6,7,Suministros de Oficina,Henderson,United States,Maria Etezadi,Rogers,2019-01-06,US-2019-167199,42420,Rogers Handheld Barrel Pencil Sharpener,Sur,Home Office,2019-01-10,Kentucky,0.0,1.4796,2,5.480
7,8,Suministros de Oficina,Athens,United States,Jack O'Briant,Dixon,2019-01-06,US-2019-106054,30605,"Dixon Prang Watercolor Pencils, 10-Color Set w...",Sur,Corporativo,2019-01-07,Georgia,0.0,5.2398,3,12.780
8,9,Suministros de Oficina,Henderson,United States,Maria Etezadi,Ibico,2019-01-06,US-2019-167199,42420,Ibico Hi-Tech Manual Binding System,Sur,Home Office,2019-01-10,Kentucky,0.0,274.4910,2,609.980
9,10,Suministros de Oficina,Henderson,United States,Maria Etezadi,Alliance,2019-01-06,US-2019-167199,42420,"Alliance Super-Size Bands, Assorted Sizes",Sur,Home Office,2019-01-10,Kentucky,0.0,0.3112,4,31.120


In [ ]:
# Truncar la columna fecha_orden y luego convertir a tipo date

query="""
ALTER TABLE cohort.supertienda
    MODIFY COLUMN Fecha_de_Orden DATE,
    MODIFY COLUMN Fecha_de_envio DATE;
"""
with engine.begin() as conn:
    resultado = conn.execute(text(query))

print(f"Filas actualizadas: {resultado.rowcount}")

Filas actualizadas: 10194


In [22]:
# Obtener la fecha en que cada cliente hizo su primera orden

query="""
select
    nombre_del_cliente,
    min(fecha_de_orden) as fecha_primera_orden
from
    cohort.supertienda
group by
    1;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,nombre_del_cliente,fecha_primera_orden
0,Darren Powers,2019-01-03
1,Phillina Ober,2019-01-04
2,Mick Brown,2019-01-05
3,Maria Etezadi,2019-01-06
4,Jack O'Briant,2019-01-06
5,Lycoris Saunders,2019-01-06
6,Vivek Sundaresam,2019-01-07
7,Melanie Seite,2019-01-09
8,Anthony Jacobs,2019-01-10
9,Seth Vernon,2019-01-11
